# Video Understanding with VSS

This notebook uploads an MP4 video to the **Video Search and Summarization (VSS) agent**, completes the video ingestion workflow, and then uses the `video_understanding` skill to answer a natural-language question about a selected portion of the video.

## Setup

Setup used a brev instance - bad-bronze-antelope :
    NVIDIA RTX PRO Server 6000 (96 GB)
    4 GPUs x 96 CPUs
    1 TiB RAM

VSS deployed using dedicated GPU :

    deploy/docker/scripts/dev-profile.sh up -p base -H RTXPRO6000BW --llm-device-id 0 --vlm-device-id 1

## Workflow

The notebook performs the following steps:

1. Accepts the path to a local `.mp4` video file.
2. Accepts a natural-language `user_prompt` describing what should be identified or answered from the video.
3. Accepts an optional `start_time` and `end_time` defining the section of the video that is relevant to the question.
4. Requests a VST upload URL from the `POST /api/v1/videos` endpoint.
5. Uploads the video file directly to the returned VST URL.
6. Completes the video upload using `POST /api/v1/videos/{sensor_id}/complete`.
7. Sends the `sensor_id`, user prompt, and applicable time range to the `POST /v1/chat` endpoint using the `video_understanding` skill.
8. Parses the `/v1/chat` response and removes the internal `<agent-think>` information, returning the final video-understanding answer.

## Required Inputs

The notebook requires the following inputs:

- **`video_path`** — Path to the local MP4 video file.

  Example: `/home/ubuntu/sample-data/videoplayback.mp4`

- **`user_prompt`** — Natural-language question about the video.

  Example: `What is the color of the tie the man is wearing?`

- **`start_time`** — Start of the video section of interest, using `HH:MM:SS` format.

  Example: `00:00:08`

- **`end_time`** — End of the video section of interest, using `HH:MM:SS` format.

  This can be set to `None` when no explicit end time is required.

## Example Inputs

```python
video_path = "/home/ubuntu/sample-data/videoplayback.mp4"

user_prompt = "What is the color of the tie the man is wearing?"

start_time = "00:00:08"

end_time = None

## VSS API

The notebook communicates with the local VSS agent at:
    
    http://127.0.0.1:8000

The main endpoints used by the notebook are:

    POST /api/v1/videos
    POST /api/v1/videos/{sensor_id}/complete
    POST /v1/chat

The final request invokes the **video_understanding** skill using the video **sensor_id** and the supplied question and time range.

The final output is the answer produced by the VSS **video_understanding** skill for the specified video section.

# Configuration and Imports

In [19]:
import json
import re
from pathlib import Path

import requests
from IPython.display import display, Markdown


VSS_BASE_URL = "http://127.0.0.1:8000"
CHAT_URL = f"{VSS_BASE_URL}/v1/chat"

VIDEO_API_URL = f"{VSS_BASE_URL}/api/v1/videos"

REQUEST_TIMEOUT = 300


# User Inputs

In [20]:
# Path to the local MP4 file
video_path = "/home/ubuntu/sample-data/videoplayback.mp4"

# Natural-language question to ask about the video
# user_prompt = "What is the color of the tie the man is wearing?"
user_prompt = "How many Ja/Oui/Si votes were there?"

# Time range of interest.
# Use "HH:MM:SS" or "HH:MM:SS.sss".
#
# Set either value to None if the endpoint should receive no corresponding
# time constraint.
# start_time = "00:00:08"
start_time = "00:04:18"
end_time = None


# Validate Video Input

In [18]:
video_file = Path(video_path).expanduser().resolve()

if not video_file.exists():
    raise FileNotFoundError(f"Video does not exist: {video_file}")

if not video_file.is_file():
    raise ValueError(f"Path is not a file: {video_file}")

if video_file.suffix.lower() != ".mp4":
    raise ValueError(f"Expected an MP4 file, got: {video_file.suffix}")

print(f"Video: {video_file}")
print(f"Size: {video_file.stat().st_size:,} bytes")

Video: /home/ubuntu/sample-data/videoplayback.mp4
Size: 33,730,466 bytes


# Time parsing helpers

In [4]:
def time_to_seconds(value):
    """
    Convert HH:MM:SS, HH:MM:SS.sss, MM:SS, or seconds to float seconds.
    """
    if value is None:
        return None

    if isinstance(value, (int, float)):
        return float(value)

    value = str(value).strip()

    parts = value.split(":")

    if len(parts) == 3:
        hours, minutes, seconds = parts
        return (
            int(hours) * 3600
            + int(minutes) * 60
            + float(seconds)
        )

    if len(parts) == 2:
        minutes, seconds = parts
        return int(minutes) * 60 + float(seconds)

    return float(value)


start_seconds = time_to_seconds(start_time)
end_seconds = time_to_seconds(end_time)

if start_seconds is not None and start_seconds < 0:
    raise ValueError("start_time cannot be negative")

if end_seconds is not None and end_seconds < 0:
    raise ValueError("end_time cannot be negative")

if (
    start_seconds is not None
    and end_seconds is not None
    and end_seconds <= start_seconds
):
    raise ValueError("end_time must be greater than start_time")

print("start_seconds:", start_seconds)
print("end_seconds:", end_seconds)


start_seconds: 258.0
end_seconds: None


# Request the VST upload URL
This is the first API call corresponding to the flow you established:

In [5]:
response = requests.post(
    VIDEO_API_URL,
    json={
        "filename": str(video_file)
    },
    timeout=REQUEST_TIMEOUT,
)

response.raise_for_status()

upload_info = response.json()

print(json.dumps(upload_info, indent=2))


{
  "url": "http://172.31.18.126:7777/vst/api/v1/storage/file"
}


# Extract the VST upload URL

In [6]:
upload_url = upload_info.get("url")

if not upload_url:
    raise RuntimeError(
        f"VSS did not return an upload URL: {upload_info}"
    )

print(f"VST upload URL: {upload_url}")


VST upload URL: http://172.31.18.126:7777/vst/api/v1/storage/file


# Upload the MP4 to VST

In [7]:
with video_file.open("rb") as f:
    upload_response = requests.post(
        upload_url,
        files={
            "file": (
                video_file.name,
                f,
                "video/mp4",
            )
        },
        timeout=REQUEST_TIMEOUT,
    )

upload_response.raise_for_status()

print(upload_response.text)


{
	"bytes" : 33730466,
	"created_at" : "2026-9-24T6:49:34.924Z",
	"filePath" : "/home/vst/vst_release/streamer_videos/videoplayback_8.mp4",
	"filename" : "videoplayback_8",
	"id" : "747d53e8-a123-42ab-b374-b3761db123b7",
	"sensorId" : "cd52be42-a97a-4695-95fa-e8b4e22a11a1",
	"streamId" : "cd52be42-a97a-4695-95fa-e8b4e22a11a1"
}


# Parse the VST upload response

In [9]:
vst_result = upload_response.json()

sensor_id = (
    vst_result.get("sensorId")
    or vst_result.get("sensor_id")
)

filename = vst_result.get("filename") or video_file.stem

if not sensor_id:
    raise RuntimeError(
        f"VST response did not contain sensorId: {vst_result}"
    )

print(f"sensor_id: {sensor_id}")
print(f"filename: {filename}")


sensor_id: 5208b811-5596-44b9-a15e-8440eb5f6094
filename: videoplayback_1


# Complete the VSS upload

In [8]:
complete_url = (
    f"{VSS_BASE_URL}/api/v1/videos/{sensor_id}/complete"
)

complete_response = requests.post(
    complete_url,
    headers={
        "Content-Type": "application/json"
    },
    json={
        "filename": filename
    },
    timeout=REQUEST_TIMEOUT,
)

complete_response.raise_for_status()

complete_result = complete_response.json()

print(json.dumps(complete_result, indent=2))


NameError: name 'sensor_id' is not defined

# Construct the /video_understanding prompt

In [11]:
def build_video_understanding_prompt(
    sensor_id,
    user_prompt,
    start_time=None,
    end_time=None,
):
    parts = [
        "/video_understanding",
        "sensor_id",
        sensor_id,
    ]

    time_context = []

    if start_time is not None:
        time_context.append(f"start_time {start_time}")

    if end_time is not None:
        time_context.append(f"end_time {end_time}")

    if time_context:
        parts.extend(time_context)

    parts.append(user_prompt)

    return " ".join(parts)


video_understanding_prompt = build_video_understanding_prompt(
    sensor_id=sensor_id,
    user_prompt=user_prompt,
    start_time=start_time,
    end_time=end_time,
)

print(video_understanding_prompt)


/video_understanding sensor_id 5208b811-5596-44b9-a15e-8440eb5f6094 start_time 00:00:08 What is the color of the tie the man is wearing?


# Call /v1/chat

In [12]:
chat_payload = {
    "messages": [
        {
            "role": "user",
            "content": video_understanding_prompt,
        }
    ]
}

chat_response = requests.post(
    CHAT_URL,
    headers={
        "Content-Type": "application/json"
    },
    json=chat_payload,
    timeout=REQUEST_TIMEOUT,
)

chat_response.raise_for_status()

chat_result = chat_response.json()

print(json.dumps(chat_result, indent=2))


{
  "id": "7a0b769a-1069-4e95-bc5a-7542e886577e",
  "object": "chat.completion",
  "model": "unknown-model",
  "created": 1789925529,
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "\n\n<agent-think><agent-think-step title=\"1 - Thought\">Plan:   1. Call `video_understanding` \u2014 analyze the video clip from sensor_id 5208b811-5596-44b9-a15e-8440eb5f6094 starting at 00:00:08 (8 seconds) to determine the color of the tie the man is wearing. Include the user's query in the `user_prompt` to focus on the tie's color.</agent-think-step>\n<agent-think-step title=\"2 - Tool Call\">Tool: video_understanding Args: {'user_prompt': 'What is the color of the tie the man is wearing?', 'sensor_id': '5208b811-5596-44b9-a15e-8440eb5f6094', 'start_timestamp': 8.0, 'vlm_reasoning': None} Result: The man is wearing a blue tie.</agent-think-step>\n<agent-think-step title=\"3 - Thought\">Updated Plan:  1. [x] Call `video_understanding` \u2014 a

# Extract the assistant message

In [13]:
choices = chat_result.get("choices", [])

if not choices:
    raise RuntimeError(
        f"/v1/chat returned no choices: {chat_result}"
    )

message = choices[0].get("message", {})
raw_content = message.get("content", "")

if not raw_content:
    raise RuntimeError(
        f"/v1/chat returned an empty message: {chat_result}"
    )

print(raw_content)




<agent-think><agent-think-step title="1 - Thought">Plan:   1. Call `video_understanding` — analyze the video clip from sensor_id 5208b811-5596-44b9-a15e-8440eb5f6094 starting at 00:00:08 (8 seconds) to determine the color of the tie the man is wearing. Include the user's query in the `user_prompt` to focus on the tie's color.</agent-think-step>
<agent-think-step title="2 - Tool Call">Tool: video_understanding Args: {'user_prompt': 'What is the color of the tie the man is wearing?', 'sensor_id': '5208b811-5596-44b9-a15e-8440eb5f6094', 'start_timestamp': 8.0, 'vlm_reasoning': None} Result: The man is wearing a blue tie.</agent-think-step>
<agent-think-step title="3 - Thought">Updated Plan:  1. [x] Call `video_understanding` — analyzed video clip from sensor_id 5208b811-5596-44b9-a15e-8440eb5f6094 at 00:00:08; result: The man is wearing a blue tie.  --- ### Latest Tool Results `video_understanding` result: The man is wearing a blue tie.</agent-think-step></agent-think>

The color of the

# Parse out <agent-think> and the final answer

In [14]:
def extract_final_answer(content):
    """
    Remove <agent-think>...</agent-think> from the chat response
    and return the remaining user-facing answer.
    """
    if not content:
        return ""

    # Remove agent-think blocks, including multiline content.
    cleaned = re.sub(
        r"<agent-think>.*?</agent-think>",
        "",
        content,
        flags=re.DOTALL | re.IGNORECASE,
    )

    # Normalize whitespace.
    cleaned = cleaned.strip()

    return cleaned


final_answer = extract_final_answer(raw_content)

print(final_answer)


The color of the tie the man is wearing is **blue**.


# Display the final result

In [15]:
display(Markdown(
    f"""
### Video Understanding Result

**Video:** `{video_file.name}`

**Sensor ID:** `{sensor_id}`

**Time range:** `{start_time or "beginning"}`
 → `{end_time or "end"}`

**Question:** {user_prompt}

**Answer:** {final_answer}
"""
))



### Video Understanding Result

**Video:** `videoplayback.mp4`

**Sensor ID:** `5208b811-5596-44b9-a15e-8440eb5f6094`

**Time range:** `00:00:08`
 → `end`

**Question:** What is the color of the tie the man is wearing?

**Answer:** The color of the tie the man is wearing is **blue**.


# One-cell end-to-end function

In [12]:
def analyze_video(
    video_path,
    user_prompt,
    start_time=None,
    end_time=None,
):
    video_file = Path(video_path).expanduser().resolve()

    if not video_file.exists():
        raise FileNotFoundError(video_file)

    if video_file.suffix.lower() != ".mp4":
        raise ValueError("Only MP4 files are supported")

    # 1. Get VST upload URL
    response = requests.post(
        VIDEO_API_URL,
        json={"filename": str(video_file)},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()

    upload_info = response.json()
    upload_url = upload_info["url"]

    # 2. Upload video to VST
    with video_file.open("rb") as f:
        response = requests.post(
            upload_url,
            files={
                "file": (
                    video_file.name,
                    f,
                    "video/mp4",
                )
            },
            timeout=REQUEST_TIMEOUT,
        )

    response.raise_for_status()
    vst_result = response.json()

    sensor_id = (
        vst_result.get("sensorId")
        or vst_result.get("sensor_id")
    )

    if not sensor_id:
        raise RuntimeError(
            f"No sensorId in VST response: {vst_result}"
        )

    filename = vst_result.get(
        "filename",
        video_file.stem,
    )

    # 3. Complete upload
    complete_url = (
        f"{VSS_BASE_URL}/api/v1/videos/{sensor_id}/complete"
    )

    response = requests.post(
        complete_url,
        json={"filename": filename},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()

    # 4. Construct video_understanding request
    prompt = build_video_understanding_prompt(
        sensor_id=sensor_id,
        user_prompt=user_prompt,
        start_time=start_time,
        end_time=end_time,
    )

    # 5. Ask the VSS agent
    response = requests.post(
        CHAT_URL,
        headers={
            "Content-Type": "application/json"
        },
        json={
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ]
        },
        timeout=REQUEST_TIMEOUT,
    )

    response.raise_for_status()

    chat_result = response.json()

    # 6. Extract and parse final response
    raw_content = chat_result["choices"][0]["message"]["content"]
    final_answer = extract_final_answer(raw_content)

    return {
        "sensor_id": sensor_id,
        "filename": filename,
        "prompt": prompt,
        "raw_response": chat_result,
        "answer": final_answer,
    }


# Run the complete workflow

In [15]:
result = analyze_video(
    video_path="/home/ubuntu/sample-data/videoplayback.mp4",
    user_prompt="How many Ja/Oui/Si votes were there?",
    start_time="00:04:18",
    end_time=None,
)

print("Sensor ID:", result["sensor_id"])
print("Answer:", result["answer"])


NameError: name 'build_video_understanding_prompt' is not defined